In [0]:
from datetime import date

In [0]:
max_iterations = (date.today() - date(2026, 2, 10)).days

for _ in range(max_iterations + 1):
    result = spark.sql("""
               SELECT MIN(source_file_date) AS snapshot_date
                 FROM tibia_analytics.silver.characters_history
                WHERE source_file_date > (
                        SELECT COALESCE(MAX(last_seen_date), DATE '1997-01-07')
                          FROM tibia_analytics.silver.characters_identity
                      )
             """).collect()[0][0]

    if result is None:
        dbutils.notebook.run("load_identity_overrides_table", 0)
        dbutils.notebook.run("load_history_enriched_table", 0)
        break

    dbutils.notebook.run("load_identity_state_table", 0)

else:
  raise RuntimeError(f"Loop exhausted after {max_iterations} iterations without NULL.")